- [x] удалить строки с пустым title
- [x] разбить на леммы и морфемы
- [x] дисбаланс, будем использовать stratify 
- [x] утечек данных нет

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
import os
from razdel import tokenize
import pymorphy3
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, hinge_loss, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, ParameterGrid
from time import time

In [4]:
df = pd.read_csv("../datasets/porn_detection/train.csv")
df = df.dropna(subset=['title'])

In [5]:
df.head()

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0


In [6]:
df[(df["label"] == 0) & (df["url"].str.contains("porn"))].shape

(0, 4)

#### Заметим, что некоторые комбинации букв содержатся только в сайтах 18+ (porn, porevo)
В будущем будем все сайты с таким называнием отмечать как сайты 18+ (повысим recall)

In [7]:
# preprocessing
def tokenize_df(df):

    def lemmatize_text(text):
        tokens = [token.text for token in tokenize(text)]
        lemmas = []
        for token in tokens:
            if token.isalpha():
                parsed = morph.parse(token)[0]  # берем первый вариант разбора
                lemmas.append(parsed.normal_form)
        return " ".join(lemmas)
    
    df['tokens'] = df['title'].apply(lambda x: " ".join([token.text for token in tokenize(x)])) 
    morph = pymorphy3.MorphAnalyzer()   # эта штука может убирать названия фильмов и тд
    df['lemmatized'] = df['tokens'].apply(lemmatize_text)

In [10]:
tokenize_df(df)

In [11]:
encoder = TfidfVectorizer(max_df=0.7,
                          min_df=5,
                          ngram_range=(1, 3))
X = df["lemmatized"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)
print(np.unique(y_train))
print(X_test.shape)

[0 1]
(27062, 39175)


### Fit

In [12]:
n_samples = X_train.shape[0]
version = 2
log_filename = f"log_reg_train_fit_loggin_{version}.log"

os.makedirs("plots_fit", exist_ok=True)

logger = logging.getLogger("sgd_training_fit_2")
logger.setLevel(logging.INFO)
if not logger.handlers:
    file_handler = logging.FileHandler("log_reg_train_fit_loggin.log", mode="a", encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)

param_grid = {
    "loss": ["hinge", "log_loss"], 
    "penalty" : ["l1", "l2"],
    "alpha": [1e-5, 1e-6],
    "learning_rate": ["optimal"],
    "eta0": [1e-3, 1e-4], 
    "n_iter_no_change": [5, 10], 
    "tol": [1e-4, 1e-5],
    "max_iter": [1000, 2000],
    "validation_fraction": [0.1, 0.2]
}

model_num = 0
for params in ParameterGrid(param_grid):
    time_start = time()
    model = SGDClassifier(**params, random_state = 42,
        class_weight = dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train)))),
        early_stopping = True)

    print(f"Model num: {model_num}")
    logger.info(
        f"=== New model #{model_num} "
        f"params={params} ==="
    )

    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    if params["loss"] == "log_loss": 
        cur_loss_train = log_loss(y_train, model.predict_proba(X_train))
        cur_loss_test = log_loss(y_test, model.predict_proba(X_test))
    else:
        cur_loss_train = hinge_loss(y_train, model.decision_function(X_train))
        cur_loss_test = hinge_loss(y_test, model.decision_function(X_test))
    logger.info(
        f"Model {model_num}| "
        f"f1={f1_score(y_test, y_pred_test):.4f} recall={recall_score(y_test, y_pred_test):.4f} loss_train={cur_loss_train:.4f} "
        f"loss_test={cur_loss_test:.4f}"
    )

    logger.info(f"Model {model_num} | training finished | total_elapsed={(time() - time_start):.2f}s")
    model_num+=1




Model num: 0
Model num: 1
Model num: 2
Model num: 3
Model num: 4
Model num: 5
Model num: 6
Model num: 7
Model num: 8
Model num: 9
Model num: 10
Model num: 11
Model num: 12
Model num: 13
Model num: 14
Model num: 15
Model num: 16
Model num: 17
Model num: 18
Model num: 19
Model num: 20
Model num: 21
Model num: 22
Model num: 23
Model num: 24
Model num: 25
Model num: 26
Model num: 27
Model num: 28
Model num: 29
Model num: 30
Model num: 31
Model num: 32
Model num: 33
Model num: 34
Model num: 35
Model num: 36
Model num: 37
Model num: 38
Model num: 39
Model num: 40
Model num: 41
Model num: 42
Model num: 43
Model num: 44
Model num: 45
Model num: 46
Model num: 47
Model num: 48
Model num: 49
Model num: 50
Model num: 51
Model num: 52
Model num: 53
Model num: 54
Model num: 55
Model num: 56
Model num: 57
Model num: 58
Model num: 59
Model num: 60
Model num: 61
Model num: 62
Model num: 63
Model num: 64
Model num: 65
Model num: 66
Model num: 67
Model num: 68
Model num: 69
Model num: 70
Model num: 71
Mo